## Topic: RunnableSequence

### Agenda
- 1. Runnable Type

- 2. Introduction of RunnableSequence 

- 3. Practical Example of RunnableSequence

### 1. Runnable Type
- Runnable standardization is what makes LangChain workflows composable

- Type of Runnable:
    - 1. Task Specific Runnables

    - 2. Runnable Primitives

####  1. Task Specific Runnables
- Definition:
    - These are core LangChain components that have been converted into Runnable so they can be use in pipeline.

- Purpose:
    - Perform task-specific operations like LLM calls, prompting, retrieval etc.

- Examples:
    - ChatOpenAI -> Runs an LLM model.
    - PromptTemplate -> Formats prompts dynamically.
    - Retriever -> Retrieves relevant document.

    - these are components as well as task specific runnable so that we connect or interact to another component easily.

#### 2. Runnable Primitives

- Definition:
    - These are fundamental building blocks for structuring execution logic in AI workflows.

- Purpose:
    - They help orchestrate execution by defining how different Runnable interact(sequentially, in parallel, conditionally, and so on..)

- Example:
    - 1. RunnableSequence -> Runs steps in order (use | operator)
    
    - 2. RunnableParallel -> Runs multiple steps simultaneously.

    - 3. RunnableBranch -> Implements conditional execution (if-else logic)

    - 4. RunnableLambda -> Wraps custom python functions into Runnables.

    - 5. RunnablePassthrough -> Just forwards input as output(acts as a placeholder) 

    - 6. RunnableMap -> Maps the same input across multiple functions

### 2. Introduction of RunnableSequence

- Definition:
    - A RunnableSequence is a composition of multiple Runnable components that execute one after another.

    - RunnableSequence is a sequential chain of runnables in LangChain that executes each step one after another, passing the output of one step as the input to the next step.

    - It is useful when we need to compose multiple runnables together in a structured workflow.


- Sequential Runnable = A straight pipeline.


In [ ]:
"""     - How It Works Internally

            Input Steps Flow:

                Input (Dict/Object)
                    │
                    ▼
            ┌───────────────────┐
            │  Runnable Step 1  │  ← Prompt (Takes dict, outputs Messages)
            └───────────────────┘
                    │ (Output passed directly)
                    ▼
            ┌───────────────────┐
            │  Runnable Step 2  │  ← Model (Takes Messages, outputs AIMessage)
            └───────────────────┘
                    │ (Output passed directly)
                    ▼
            ┌───────────────────┐
            │  Runnable Step 3  │  ← Parser (Takes AIMessage, outputs String)
            └───────────────────┘
                    │
                    ▼
                Final Output (String)


"""

In [ ]:
# syntax of RunnableSequence
"""  
chain = RunnableSequence(
    pass the Runnable or component that we want to sequential connect

    # eg:
    prompt, model , parser

    # model way
    prompt | model | parser


)
- here, the output prompt is automatically input of the model, similarly for the output of model is the input of parser and the output of parser is the final output that store in chain.

"""

### 3. Practical Example of RunnableSequence

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence

from dotenv import load_dotenv
load_dotenv()


# Define prompt
prompt = PromptTemplate(
    template = "write a joke about {topic}",
    input_variables=["topic"]
)

# define model
model = ChatOpenAI()

# Define parser
parser = StrOutputParser()

# Create a Sequential chain using RunnableSequence
chain = RunnableSequence(prompt, model, parser)

# model way to create chain (LangChain Expression Language (LCEL))
# chain = RunnableSequence(prompt | model | parser)


# response
response = chain.invoke(
    {
        "topic": "GenAI"
    }
)

print(f"Response: \n {response}")

In [ ]:
"""   - Most Important Runnable Methods

┌──────────────────────────────────────────┐
│             Runnable Methods             │
├──────────────────────────────────────────┤
│ invoke() → execute one input             │
├──────────────────────────────────────────┤
│ batch()  → process multiple inputs       │
├──────────────────────────────────────────┤
│ stream() → receive incremental output    │
├──────────────────────────────────────────┤
│ bind()   → bind invocation parameters    │
├──────────────────────────────────────────┤
│ with_config() → configure execution      │
├──────────────────────────────────────────┤
│ with_retry() → retry failures            │
├──────────────────────────────────────────┤
│ with_fallbacks() → fallback workflows    │
└──────────────────────────────────────────┘

"""


""" 
- 1. chain.invoke(input_1):
    - invoke() is the most fundamental execution method.
    - Execute the Runnable with one input.

    
- 2. results = chain.batch([
        input_1,
        input_2,
        input_3
    ]) 

    - topics = [
        {"topic": "Machine Learning"},
        {"topic": "Deep Learning"},
        {"topic": "Generative AI"}
    ]

    - Use:
        - multiple inputs


- 3. chain.batch(inputs):
    - Actual speedup depends on the Runnable/provider and how concurrency is configured.
    - This is especially useful for:
        - document processing
        - data extraction
        - classification
        - summarization
        - evaluation
        - embedding workflows
        - bulk LLM processing    
            
- 4. for chunk in model.stream(
    "Explain Generative AI"
):
    print(chunk.content, end="", flush=True)
    
    use :
    - The model produces chunks progressively
    - Sometimes we don't want to wait for the complete response
"""



In [ ]:
# Example 2: Create an {topic} and explain the {topic}

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence

from dotenv import load_dotenv
load_dotenv()


# Define prompt
prompt1 = PromptTemplate(
    template = "write a joke about {topic}",
    input_variables=["topic"]
)

prompt2 = PromptTemplate(
    template = "Explain the following - {topic}",
    input_variables=["topic"]
)

# define model
model = ChatOpenAI()

# Define parser
parser = StrOutputParser()

# Create a Sequential chain using RunnableSequence with (LangChain Expression Language (LCEL)

chain = RunnableSequence(prompt1 | model | parser | prompt2 | model | parser)


# response
response = chain.invoke(
    {
        "topic": "GenAI"
    }
)

print(f"Response: \n {response}")

### Summary
- A Sequential Runnable (officially RunnableSequence) is a LangChain component that runs multiple Runnables one after another, where the output of each step automatically becomes the input of the next step.

